# Exploratory Data Analysis Assignment
### Superstore Dataset — HeroVired / HeroX

**Student:** Mani Dixit | **Batch:** PGDSAI3
---

## Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid', palette='Set2')

df_raw = pd.read_csv('../data/raw/superstore.csv')
df_raw.head()

---
## Q1: Data Cleaning

### Q1 i — Count and Handle Missing Values

In [ ]:
print('Missing values per column:')
print(df_raw.isnull().sum())
print(f'\nTotal missing cells: {df_raw.isnull().sum().sum()}')

**Q1 i — Insight:**  
There are **5 missing values** in `Shipping.Cost` and **3 missing values** in `State`.  
Since both are small counts (~0.01% of 51,290 rows), we drop those rows rather than impute, to avoid introducing noise.

In [ ]:
df = df_raw.dropna().reset_index(drop=True)
print(f'Shape after dropping missing rows: {df.shape}')
print(f'Remaining missing values: {df.isnull().sum().sum()}')

### Q1 ii — Deal with Duplicate Values

In [ ]:
dups = df.duplicated().sum()
print(f'Duplicate rows: {dups}')

**Q1 ii — Insight:** No duplicate rows were found in the dataset. No action required.

### Q1 iii — Delete Unknown Columns

In [ ]:
print('Unknown column value counts:')
print(df['记录数'].value_counts())
df.drop(columns=['记录数'], inplace=True)
print(f'\nShape after dropping unknown column: {df.shape}')

**Q1 iii — Insight:** The column `记录数` (ji_lu_shu) contains only the constant value **1** — zero analytical information. Dropped.

### Q1 iv — Shape, Size, and Datatypes

In [ ]:
print('Shape (rows, cols):', df.shape)
print('Size (total elements):', df.size)
print('\nData Types:')
print(df.dtypes)

**Q1 iv — Insight:**  
- **Shape:** 51,282 rows × 26 columns (after cleaning).  
- **Numerical:** `Discount`, `Profit`, `Quantity`, `Row.ID`, `Sales`, `Shipping.Cost`, `Year`, `weeknum`.  
- **Categorical:** `Category`, `City`, `Country`, `Customer.ID`, `Customer.Name`, `Market`, `Order.Date`, `Order.ID`, `Order.Priority`, `Product.ID`, `Product.Name`, `Region`, `Segment`, `Ship.Date`, `Ship.Mode`, `State`, `Sub.Category`, `Market2`.  
- Date columns are stored as strings and should be parsed for time-series analysis.

---
## Q2: Univariate Analysis — Numerical Features

In [ ]:
num_cols = ['Discount', 'Profit', 'Quantity', 'Row.ID', 'Sales', 'Shipping.Cost', 'Year', 'weeknum']
print(df[num_cols].describe().round(2))

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')
plt.suptitle('Q2 — Univariate Distributions of Numerical Features', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('q2_univariate_num.png', bbox_inches='tight')
plt.show()

### Q2 i — Which features seem useless?

**Q2 i — Insight:**  
- **`Row.ID`** is a sequential integer index (1 → 51,290). It carries no business meaning and should be excluded from all analysis.  
- **`Year`** has only 4 distinct values (2011–2014) and is better treated as an ordinal categorical variable.

### Q2 ii — Uniformly or normally distributed features?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ['Quantity', 'weeknum', 'Discount']):
    ax.hist(df[col], bins=30, color='mediumseagreen', edgecolor='white', alpha=0.85)
    ax.set_title(f'{col}', fontweight='bold')
    ax.set_xlabel(col); ax.set_ylabel('Frequency')
plt.suptitle('Q2 ii — Near-Uniform / Near-Normal Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('q2_uniform_normal.png', bbox_inches='tight')
plt.show()

**Q2 ii — Insight:**  
- **`weeknum`** is approximately **uniformly distributed** — orders are placed fairly evenly across all weeks.  
- **`Quantity`** is close to **uniform** with a slight right lean.  
- **`Discount`** is discrete multi-modal — neither uniform nor normal.

### Q2 iii — Right-skewed / Left-skewed features?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ['Sales', 'Shipping.Cost', 'Profit']):
    ax.hist(df[col], bins=60, color='salmon', edgecolor='white', alpha=0.85)
    ax.set_title(f'{col} (skew={df[col].skew():.2f})', fontweight='bold')
    ax.set_xlabel(col); ax.set_ylabel('Frequency')
plt.suptitle('Q2 iii — Skewed Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('q2_skew.png', bbox_inches='tight')
plt.show()

**Q2 iii — Insight:**  
- **`Sales`** and **`Shipping.Cost`** are strongly **right-skewed**: most orders are small, but a few high-value orders pull the mean far above the median.  
- **`Profit`** is right-skewed overall but has a visible **left tail** (losses), indicating heavy discounting erodes margins.

### Q2 iv — Features with High Outliers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ['Sales', 'Profit', 'Shipping.Cost']):
    ax.boxplot(df[col], vert=True, patch_artist=True,
               boxprops=dict(facecolor='lightcoral', color='darkred'),
               medianprops=dict(color='black', linewidth=2))
    ax.set_title(f'{col}', fontweight='bold')
    ax.set_ylabel(col)
plt.suptitle('Q2 iv — Outlier Detection (Box Plots)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('q2_outliers.png', bbox_inches='tight')
plt.show()

for col in ['Sales', 'Profit', 'Shipping.Cost']:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(f'{col}: {n_out} outliers ({n_out/len(df)*100:.1f}%)')

**Q2 iv — Insight:**  
- **`Sales`**, **`Profit`**, and **`Shipping.Cost`** all have high outliers in the upper tail.  
- **`Profit`** also has significant lower outliers (large losses), especially in Furniture where heavy discounts erode margins.  
- For modeling, log-transformation or winsorization should be applied. Median and IQR are preferred over mean and std.

---
## Q3: Univariate Analysis — Categorical Features

In [ ]:
cat_cols = ['Category', 'Segment', 'Ship.Mode', 'Order.Priority', 'Market', 'Region', 'Sub.Category']
for col in cat_cols:
    print(f'{col}: {df[col].nunique()} unique values')

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    vc = df[col].value_counts()
    axes[i].barh(vc.index[:15], vc.values[:15], color=sns.color_palette('Set2', len(vc))[:15])
    axes[i].set_title(f'{col}', fontweight='bold')
    axes[i].set_xlabel('Count')
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Q3 — Univariate Analysis of Categorical Features', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('q3_categorical.png', bbox_inches='tight')
plt.show()

### Q3 i — Inaccurate / Not Useful Categorical Features

**Q3 i — Insight:**  
- **`Order.ID`**, **`Product.ID`**, **`Customer.ID`**, **`Row.ID`** are unique identifiers — no categorical insight value.  
- **`Order.Date`** and **`Ship.Date`** are stored as strings; must be parsed before temporal analysis.  
- **`Product.Name`** has thousands of unique values; too granular for direct categorical insight.

### Q3 ii — Issue with Customer.Name as a Categorical Feature

**Q3 ii — Insight:**  
`Customer.Name` has ~1,590 unique values. Using it as a categorical feature would:  
1. Cause **high cardinality** → one-hot encoding explodes to 1,590+ columns.  
2. Lead to **data leakage** — names encode customer-level history, not generalizable rules.  
3. Cause **overfitting** — model learns individual quirks rather than segment patterns.

### Q3 iii — Is Category Distribution Balanced or Skewed?

In [ ]:
print(df['Category'].value_counts())
df['Category'].value_counts().plot(kind='bar', color=['#2196F3','#FF9800','#4CAF50'], edgecolor='black', rot=0)
plt.title('Category Distribution', fontweight='bold')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('q3_category.png', bbox_inches='tight')
plt.show()

**Q3 iii — Insight:**  
The distribution is **skewed** — Office Supplies dominates with ~61% of orders, while Technology and Furniture each account for ~20%.

### Q3 iv — Does One Country Dominate?

In [ ]:
top10 = df['Country'].value_counts().head(10)
top10.plot(kind='bar', color='steelblue', edgecolor='black', rot=45)
plt.title('Top 10 Countries by Order Count', fontweight='bold')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('q3_country.png', bbox_inches='tight')
plt.show()
print(top10)
print(f'\nUnited States share: {df["Country"].value_counts()["United States"]/len(df)*100:.1f}%')

**Q3 iv — Insight:**  
**Yes — the United States dominates** with ~19.5% of all orders. This introduces **geographic bias**: any model trained on this data will be better calibrated for US patterns.

### Q3 v — Is City Dataset Concentrated or Spread Out?

In [ ]:
print(f'Total unique cities: {df["City"].nunique()}')
print(f'Top 10 cities cover: {df["City"].value_counts().head(10).sum()/len(df)*100:.1f}% of orders')
df['City'].value_counts().head(15).plot(kind='bar', color='teal', edgecolor='black', rot=45)
plt.title('Top 15 Cities by Order Count', fontweight='bold')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('q3_city.png', bbox_inches='tight')
plt.show()

**Q3 v — Insight:**  
The dataset covers **~3,636 unique cities** — highly **spread out**. However, the top 15 cities account for a disproportionate share of orders, confirming a **heavy-tailed urban concentration pattern**.

---
## Q4: Bivariate Analysis — Numerical vs Numerical

In [ ]:
num_analysis = ['Discount', 'Profit', 'Quantity', 'Sales', 'Shipping.Cost']
corr = df[num_analysis].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Q4 — Correlation Heatmap (Numerical Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('q4_heatmap.png', bbox_inches='tight')
plt.show()
print(corr)

### Q4 i — Most Strongly Correlated Features

**Q4 i — Insight:**  
**`Sales` and `Shipping.Cost`** are the most strongly positively correlated pair (**r = 0.77**). Larger orders involve bulkier shipments, resulting in higher shipping costs.

### Q4 ii — Negatively Correlated Features

**Q4 ii — Insight:**  
- **`Discount` and `Profit`**: **r = -0.32** — strongest negative correlation. Higher discounts directly erode margins.  
- **`Discount` and `Sales`**: **r = -0.09** — weak negative.  
- **`Discount` and `Shipping.Cost`**: **r = -0.08** — very weak negative.

### Q4 iii — Bivariate Checks for Profit

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].scatter(df['Sales'], df['Profit'], alpha=0.2, color='steelblue', s=5)
axes[0].set_xlabel('Sales'); axes[0].set_ylabel('Profit')
axes[0].set_title('Profit vs Sales', fontweight='bold')
axes[1].scatter(df['Discount'], df['Profit'], alpha=0.2, color='salmon', s=5)
axes[1].set_xlabel('Discount'); axes[1].set_ylabel('Profit')
axes[1].set_title('Profit vs Discount', fontweight='bold')
axes[2].scatter(df['Shipping.Cost'], df['Profit'], alpha=0.2, color='mediumseagreen', s=5)
axes[2].set_xlabel('Shipping.Cost'); axes[2].set_ylabel('Profit')
axes[2].set_title('Profit vs Shipping Cost', fontweight='bold')
plt.suptitle('Q4 iii — Profit Bivariate Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('q4_profit_bivariate.png', bbox_inches='tight')
plt.show()

**Q4 iii — Insights:**  
- **Profit vs Sales (r=0.48):** Positive but noisy — higher sales tend to generate more profit, but not always.  
- **Profit vs Discount (r=-0.32):** Orders with discounts above ~40% almost universally generate losses.  
- **Profit vs Shipping Cost (r=0.35):** Higher-cost shipments accompany higher-value orders that also produce more profit.

### Q4 iv — Time Effects

In [ ]:
yearly = df.groupby('Year')[['Sales', 'Profit']].sum().reset_index()
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(yearly['Year'], yearly['Sales'], color='steelblue', alpha=0.7, label='Total Sales')
ax1.set_xlabel('Year'); ax1.set_ylabel('Total Sales', color='steelblue')
ax2 = ax1.twinx()
ax2.plot(yearly['Year'], yearly['Profit'], color='crimson', marker='o', linewidth=2.5, label='Total Profit')
ax2.set_ylabel('Total Profit', color='crimson')
ax1.set_title('Q4 iv — Sales and Profit Trend by Year', fontsize=13, fontweight='bold')
fig.legend(loc='upper left', bbox_to_anchor=(0.1, 0.88))
plt.tight_layout()
plt.savefig('q4_time.png', bbox_inches='tight')
plt.show()
print(yearly)

**Q4 iv — Insight:**  
Both **Sales and Profit grow consistently year-over-year** from 2011 to 2014 — Sales nearly doubled (~$2.26M → $4.30M) and Profit roughly doubled (~$249K → $504K). Stable linear growth with no visible seasonal breaks.

---
## Q5: Bivariate Analysis — Categorical vs Numerical

### Q5 i — Profit by Category

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='Category', y='Profit', ax=axes[0], palette='Set2', showfliers=True,
            flierprops=dict(marker='o', markersize=2, alpha=0.3))
axes[0].set_title('Profit by Category (with outliers)', fontweight='bold')
axes[0].set_ylim(-500, 1000)
sns.boxplot(data=df, x='Category', y='Profit', ax=axes[1], palette='Set2', showfliers=False)
axes[1].set_title('Profit by Category (IQR focus)', fontweight='bold')
plt.suptitle('Q5 i — Profit by Category', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('q5_profit_category.png', bbox_inches='tight')
plt.show()
for cat in df['Category'].unique():
    sub = df[df['Category']==cat]['Profit']
    print(f'{cat}: median={sub.median():.2f}, IQR={sub.quantile(0.75)-sub.quantile(0.25):.2f}')

**Q5 i — Insights:**  
- **Highest median profit:** Technology (~$29.94).  
- **Lowest median profit:** Office Supplies (~$6.55) — high volume but thin margins.  
- **Widest IQR:** Technology (IQR ≈ $98.35) — most inconsistent profits.  
- **Furniture** has many negative-profit orders, suggesting deep discounting regularly destroys margins.

### Q5 ii — Sales by Category

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='Category', y='Sales', ax=axes[0], palette='Set2', showfliers=False)
axes[0].set_title('Sales by Category (IQR focus)', fontweight='bold')
medians = df.groupby('Category')[['Sales','Profit']].median().reset_index()
x = range(len(medians)); width = 0.35
axes[1].bar([i - width/2 for i in x], medians['Sales'], width, label='Median Sales', color='steelblue')
axes[1].bar([i + width/2 for i in x], medians['Profit'], width, label='Median Profit', color='salmon')
axes[1].set_xticks(list(x)); axes[1].set_xticklabels(medians['Category'])
axes[1].legend(); axes[1].set_title('Median Sales vs Profit by Category', fontweight='bold')
plt.suptitle('Q5 ii — Sales by Category', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('q5_sales_category.png', bbox_inches='tight')
plt.show()
print(medians)

**Q5 ii — Insights:**  
- **Highest median sales:** Technology (~$260).  
- **Yes** — Technology also has the highest median profit, confirming premium pricing yields both higher revenue and better margins.  
- Furniture has second-highest sales but far lower profit, revealing a margin inefficiency.

### Q5 iii — Profit by Segment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='Segment', y='Profit', ax=axes[0], palette='Pastel1',
            flierprops=dict(marker='o', markersize=2, alpha=0.3))
axes[0].set_ylim(-600, 800)
axes[0].set_title('Profit by Segment (full range)', fontweight='bold')
sns.boxplot(data=df, x='Segment', y='Profit', ax=axes[1], palette='Pastel1', showfliers=False)
axes[1].set_title('Profit by Segment (IQR focus)', fontweight='bold')
plt.suptitle('Q5 iii — Profit by Segment', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('q5_profit_segment.png', bbox_inches='tight')
plt.show()
print(df.groupby('Segment')['Profit'].describe())

**Q5 iii — Insights:**  
- **Highest median profit:** Home Office ($9.32) — marginally above Corporate and Consumer.  
- **Most negative outliers:** **Consumer** segment — largest order volume with widest spread of extreme losses from promotional discounting.

### Q5 iv — Sales by Segment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='Segment', y='Sales', ax=axes[0], palette='Pastel2', showfliers=False)
axes[0].set_title('Sales by Segment', fontweight='bold')
seg = df.groupby('Segment')[['Sales','Profit']].median().reset_index()
x = range(len(seg)); width = 0.35
axes[1].bar([i - width/2 for i in x], seg['Sales'], width, label='Median Sales', color='steelblue')
axes[1].bar([i + width/2 for i in x], seg['Profit'], width, label='Median Profit', color='salmon')
axes[1].set_xticks(list(x)); axes[1].set_xticklabels(seg['Segment'])
axes[1].legend(); axes[1].set_title('Median Sales vs Profit by Segment', fontweight='bold')
plt.suptitle('Q5 iv — Sales by Segment', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('q5_sales_segment.png', bbox_inches='tight')
plt.show()
print(seg)

**Q5 iv — Insights:**  
- All three segments have the **same median Sales ($85)** — order value is identical across segments at the median.  
- Segment does not strongly differentiate profitability at the median level.  
- The key difference is in the **tails**: Consumer has more extreme loss outliers despite similar median sales.

---
## Q6: Bivariate Analysis — Market vs Region, Category, Country

### Q6 i — Is Market Randomly Spread Across Regions?

In [ ]:
ct = pd.crosstab(df['Market'], df['Region'])
plt.figure(figsize=(14, 6))
sns.heatmap(ct, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.4,
            cbar_kws={'label': 'Order Count'})
plt.title('Q6 i — Market vs Region (Order Counts)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('q6_market_region.png', bbox_inches='tight')
plt.show()

**Q6 i — Insight:**  
**No — Market is NOT randomly spread across Regions.** Each Market maps to a strict, mutually exclusive set of Regions:  
- **US** → West, East, Central, South; **EU** → North, South, Central (Europe); **APAC** → Oceania, SE Asia, North/Central Asia  
- Market is essentially a higher-level aggregation of Region — using both together introduces multicollinearity.

### Q6 ii — Which Country Has Negligible Office Supply Orders?

In [ ]:
office_by_country = df[df['Category'] == 'Office Supplies']['Country'].value_counts()
negligible = office_by_country[office_by_country <= 2]
print(f'Countries with ≤2 Office Supply orders ({len(negligible)} countries):')
print(negligible.sort_values())
bottom15 = office_by_country.tail(15)
bottom15.plot(kind='barh', color='tomato', edgecolor='black')
plt.title('Q6 ii — Countries with Fewest Office Supply Orders', fontweight='bold')
plt.xlabel('Number of Orders')
plt.tight_layout()
plt.savefig('q6_office_country.png', bbox_inches='tight')
plt.show()

**Q6 ii — Insight:**  
Countries like **Armenia, Eritrea, Swaziland, Bahrain, Tajikistan** each have only 1–2 Office Supply orders — effectively negligible. These are small/under-penetrated markets where category-level conclusions are statistically unreliable.

### Q6 iii — Most Useful Market Insights

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
market_profit = df.groupby('Market')['Profit'].sum().sort_values(ascending=False).reset_index()
axes[0].bar(market_profit['Market'], market_profit['Profit'], color=sns.color_palette('Set2', 7))
axes[0].set_title('Total Profit by Market', fontweight='bold')
axes[0].set_ylabel('Total Profit ($)')
axes[0].tick_params(axis='x', rotation=30)
mc = df.groupby(['Market', 'Category']).size().unstack(fill_value=0)
mc.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2', edgecolor='white')
axes[1].set_title('Order Volume by Market & Category', fontweight='bold')
axes[1].set_ylabel('Order Count')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.suptitle('Q6 iii — Market-Level Insights', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('q6_market_insights.png', bbox_inches='tight')
plt.show()
print('Total Profit by Market:')
print(df.groupby('Market')['Profit'].sum().sort_values(ascending=False).round(2))

**Q6 iii — Most Useful Insights:**  
1. **APAC and EU are the top profit-generating markets** despite US having the most orders — indicating better margins in Asia-Pacific and Europe.  
2. **Canada is the smallest market** by both volume (384 orders) and profit — a niche or underdeveloped region.  
3. **Office Supplies dominate every market** by order count, but Technology drives disproportionate profit — consistent across geographies.  
4. **Market ↔ Region one-to-one mapping** means either feature can be used for geographic segmentation; Market is simpler (7 values vs 13 Regions).

---
## Summary of Key EDA Findings

| Question | Key Insight |
|---|---|
| Q1 | 8 missing rows dropped, 0 duplicates, `记录数` column dropped (constant=1) |
| Q2 | Sales, Profit, Shipping.Cost are right-skewed with significant outliers; Discount negatively impacts Profit |
| Q3 | Category is class-imbalanced (Office Supplies 61%); US dominates geographically (19.5%); 3,636 unique cities |
| Q4 | Sales–ShippingCost most correlated (r=0.77); Discount–Profit most negatively correlated (r=-0.32); YoY growth |
| Q5 | Technology has highest median sales AND profit; all segments have equal median sales ($85) |
| Q6 | Market strictly maps to Region; several small countries have negligible Office Supply orders; APAC & EU lead profit |

---
*EDA completed — Mani Dixit | PGDSAI3*